# 01 — Data Collection & Cleaning
## Nigeria Weather Risk Analysis — Abagana, Anambra State

**Project:** Seasonal Weather Risk Intelligence for Event Planning  
**Author:** [Your Name]  
**Date:** April 2026  

---

### Objective
This notebook covers:
1. Sourcing climate data for Abagana, Anambra State from NASA POWER API
2. Loading and validating the data
3. Cleaning and structuring the dataset
4. Exporting processed data for analysis and Power BI visualisation

---

### 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully')

### 2. Location Parameters

Abagana, Anambra State coordinates.

In [ ]:
# Abagana, Anambra State, Nigeria
LAT  = 6.184
LON  = 6.977
LOCATION = 'Abagana, Anambra State, Nigeria'

print(f'Location: {LOCATION}')
print(f'Coordinates: {LAT}°N, {LON}°E')

### 3. Data Collection — NASA POWER API

NASA POWER provides free access to satellite-derived meteorological data.
We will pull daily data for key climate variables.

In [ ]:
# NASA POWER API endpoint
# Documentation: https://power.larc.nasa.gov/docs/services/api/

BASE_URL = 'https://power.larc.nasa.gov/api/temporal/daily/point'

params = {
    'parameters': 'T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN',
    'community': 'RE',
    'longitude': LON,
    'latitude': LAT,
    'start': '20200101',
    'end': '20251231',
    'format': 'JSON'
}

# Parameter key:
# T2M_MAX          — Daily maximum temperature at 2m (°C)
# T2M_MIN          — Daily minimum temperature at 2m (°C)
# PRECTOTCORR      — Corrected total precipitation (mm/day)
# RH2M             — Relative humidity at 2m (%)
# ALLSKY_SFC_SW_DWN — Solar radiation (sunshine proxy, MJ/m²/day)

print('Parameters set. Run next cell to fetch data.')
print(f'Date range: 2020-01-01 to 2025-12-31')

In [ ]:
# Fetch data from NASA POWER
# Note: This may take 30–60 seconds

print('Fetching data from NASA POWER API...')
response = requests.get(BASE_URL, params=params)

if response.status_code == 200:
    raw_data = response.json()
    print('Data fetched successfully')
    print(f'Status: {response.status_code}')
else:
    print(f'Error: {response.status_code}')
    print(response.text)

### 4. Parse & Structure the Data

In [ ]:
# Extract the parameter data from JSON response
properties = raw_data['properties']['parameter']

# Build a DataFrame
df = pd.DataFrame(properties)
df.index = pd.to_datetime(df.index, format='%Y%m%d')
df.index.name = 'date'

# Rename columns for clarity
df.columns = ['temp_max_c', 'temp_min_c', 'rainfall_mm', 'humidity_pct', 'solar_rad']

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
df.head(10)

### 5. Data Cleaning

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# NASA POWER uses -999 as a fill value for missing data
df = df.replace(-999, np.nan)

print(f'\nMissing after replacing -999:')
print(df.isnull().sum())

In [ ]:
# Add derived columns
df['year']       = df.index.year
df['month']      = df.index.month
df['month_name'] = df.index.strftime('%B')
df['week']       = df.index.isocalendar().week
df['day_of_week']= df.index.day_name()
df['is_rainy_day']= (df['rainfall_mm'] > 1.0).astype(int)  # >1mm = rainy day

print('Derived columns added')
df.head()

### 6. Summary Statistics

In [ ]:
# Monthly averages — June, July, August (all years)
jja = df[df['month'].isin([6, 7, 8])]

monthly_summary = jja.groupby('month_name').agg(
    avg_temp_max   = ('temp_max_c',   'mean'),
    avg_temp_min   = ('temp_min_c',   'mean'),
    total_rainfall = ('rainfall_mm',  'sum'),
    avg_rainfall   = ('rainfall_mm',  'mean'),
    rainy_days     = ('is_rainy_day', 'sum'),
    avg_humidity   = ('humidity_pct', 'mean'),
).round(2)

print('Monthly summary (June–August, all years):')
monthly_summary

### 7. Export Processed Data

In [ ]:
# Export to processed data folder
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

# Save full cleaned dataset
df.to_csv('../data/processed/abagana_climate_daily_2020_2025.csv')

# Save JJA (June-July-August) subset
jja.to_csv('../data/processed/abagana_jja_2020_2025.csv')

# Save monthly summary
monthly_summary.to_csv('../data/processed/abagana_monthly_summary.csv')

print('Data exported successfully')
print('Files saved to ../data/processed/')

---
### Next Steps
- `02_exploratory_analysis.ipynb` — Visualise distributions and patterns
- `03_risk_model.ipynb` — Build the weekend risk scoring model
- Power BI dashboard — Load processed CSVs for interactive visualisation

---
*Nigeria Weather Risk Analysis | MSc Business Analytics Portfolio Project | April 2026*